# 第3章：深度学习快速通道 - 交互式学习

本notebook提供深度学习基础的交互式学习体验。

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets as widgets

if not Path("notebooks/bootstrap.py").exists():
    root = Path("/content/signal-to-intelligence")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/lynnyulinlin-debug/signal-to-intelligence.git", str(root)],
            check=True,
        )
    os.chdir(root)

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from notebooks.bootstrap import load_code_module

poly_mlp = load_code_module("code/ch03_deep_learning_fast/polynomial_vs_mlp.py")

%matplotlib inline

# CNN卷积操作演示
def visualize_convolution(kernel_size=3, stride=1):
    input_size = 8
    x = np.random.randn(input_size, input_size)
    kernel = np.random.randn(kernel_size, kernel_size)

    output_size = (input_size - kernel_size) // stride + 1
    output = np.zeros((output_size, output_size))

    for i in range(output_size):
        for j in range(output_size):
            patch = x[i*stride:i*stride+kernel_size, j*stride:j*stride+kernel_size]
            output[i, j] = np.sum(patch * kernel)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    im1 = axes[0].imshow(x, cmap='viridis')
    axes[0].set_title('Input')
    plt.colorbar(im1, ax=axes[0])

    im2 = axes[1].imshow(kernel, cmap='viridis')
    axes[1].set_title('Kernel')
    plt.colorbar(im2, ax=axes[1])

    im3 = axes[2].imshow(output, cmap='viridis')
    axes[2].set_title('Output')
    plt.colorbar(im3, ax=axes[2])

    plt.tight_layout()
    plt.show()

    print(f"输入大小: {x.shape}")
    print(f"卷积核大小: {kernel.shape}")
    print(f"输出大小: {output.shape}")

kernel_slider = widgets.IntSlider(value=3, min=1, max=7, step=2, description='Kernel Size:')
stride_slider = widgets.IntSlider(value=1, min=1, max=3, step=1, description='Stride:')

widgets.interact(visualize_convolution, kernel_size=kernel_slider, stride=stride_slider)


## 多层感知机 (MLP) 演示

In [ ]:
# MLP学习非线性函数
n_samples = 200
X = np.linspace(-1, 1, n_samples).reshape(-1, 1)
y = X ** 2

model = poly_mlp.SimpleNN(
    input_dim=1,
    hidden_dim=16,
    output_dim=1,
    learning_rate=0.01,
    seed=42,
)
losses = model.train(X, y, epochs=200)
y_pred = model.forward(X).flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(X, y, 'b-', linewidth=2, label='True Function')
axes[0].plot(X, y_pred, 'r--', linewidth=2, label='MLP Prediction')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('MLP Learning Nonlinear Function')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].semilogy(losses, 'g-', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss (log scale)')
axes[1].set_title('Training Loss')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"初始损失: {losses[0]:.6f}")
print(f"最终损失: {losses[-1]:.6f}")
print(f"改进: {(1 - losses[-1]/losses[0])*100:.2f}%")
